# 00 — Full Dataset Download (Problem Statement Compliant)

Following the original problem statement:
> *'Download a sector's high-cadence data which would contain around 20-30k light curves of different stars'*
> *'A curated dataset will also be provided for different types of classifiers for already known exoplanets, false positives, and eclipsing binaries'*

This notebook:
1. Downloads the full TESS Input Catalog (TIC) target list for Sector 1
2. Downloads all available 2-minute cadence light curves from that sector
3. Downloads a curated labeled dataset from ExoFOP (planets, FPs, EBs)
4. Organizes everything for the pipeline

**Expected time: 4-8 hours for full download. Run overnight.**

## 1. Imports & Setup

In [ ]:
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import os, time, glob, warnings
from tqdm import tqdm
from astroquery.mast import Catalogs, Observations
import astropy.units as u
warnings.filterwarnings('ignore')

print(f'Lightkurve : {lk.__version__}')
print('All imports OK!')

## 2. Configuration

In [ ]:
# ── CONFIGURE THESE ───────────────────────────────────────────────────
SECTOR          = 1          # TESS Sector to download (1 = July-Aug 2018)
MAX_STARS       = 1000       # Start with 1000, increase to 20000+ for full run
                             # Full sector has ~20k 2-min cadence targets
TMAG_LIMIT      = 13.0       # Only download stars brighter than this
N_PER_CLASS     = 300        # Labeled samples per class for training
BATCH_SIZE      = 50         # Download in batches to avoid server timeout
SLEEP_BETWEEN   = 1.0        # Seconds between downloads
# ─────────────────────────────────────────────────────────────────────

RAW_DIR         = '../data/raw/'
LABELED_DIR     = '../data/labeled/'
PROCESSED_DIR   = '../data/processed/'
LOG_DIR         = '../outputs/'

for d in [RAW_DIR, LABELED_DIR, PROCESSED_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Sector         : {SECTOR}')
print(f'Max stars      : {MAX_STARS}')
print(f'Tmag limit     : {TMAG_LIMIT}')
print(f'Labels/class   : {N_PER_CLASS}')
print('Directories ready!')

## 3. Get Full TIC Target List for Sector 1
Query MAST for all stars observed at 2-minute cadence in Sector 1.
This is exactly what the problem statement asks for.

In [ ]:
print(f'Querying MAST for all 2-min cadence targets in Sector {SECTOR}...')
print('This may take 1-2 minutes...')

# Search MAST for all TESS 2-min observations in Sector 1
obs_table = Observations.query_criteria(
    obs_collection = 'TESS',
    dataproduct_type = 'timeseries',
    sequence_number  = SECTOR,      # Sector number
    t_exptime        = (100, 150),  # 2-min cadence = 120 sec
)

obs_df = obs_table.to_pandas()
print(f'Total 2-min cadence observations in Sector {SECTOR}: {len(obs_df)}')

# Extract TIC IDs from target names
obs_df['tic_id'] = obs_df['target_name'].str.replace('TIC','').str.strip()
obs_df = obs_df.dropna(subset=['tic_id'])
obs_df = obs_df[obs_df['tic_id'].str.isnumeric()]

print(f'Valid TIC IDs found: {len(obs_df)}')
obs_df[['target_name','tic_id','t_exptime']].head(10)

## 4. Filter by Brightness

In [ ]:
print('Querying TIC for stellar magnitudes...')

# Get magnitudes for filtering
# Query in chunks of 1000 to avoid timeout
tic_ids_all = obs_df['tic_id'].unique().tolist()[:MAX_STARS]
print(f'Checking magnitudes for {len(tic_ids_all)} stars...')

bright_tics = []
chunk_size  = 500

for i in tqdm(range(0, min(len(tic_ids_all), 5000), chunk_size),
              desc='Querying TIC magnitudes'):
    chunk = tic_ids_all[i:i+chunk_size]
    try:
        cat = Catalogs.query_criteria(
            catalog='TIC',
            ID=chunk
        )
        cat_df = cat.to_pandas()
        bright = cat_df[cat_df['Tmag'] < TMAG_LIMIT]['ID'].astype(str).tolist()
        bright_tics.extend(bright)
    except Exception as e:
        print(f'Chunk {i} failed: {e}')
    time.sleep(0.5)

# If magnitude query fails, just use all targets
if len(bright_tics) == 0:
    print('Magnitude filter failed — using all targets')
    bright_tics = tic_ids_all

print(f'Stars passing brightness filter (Tmag < {TMAG_LIMIT}): {len(bright_tics)}')

# Save the full target list
target_df = pd.DataFrame({'tic_id': bright_tics[:MAX_STARS]})
target_df.to_csv(os.path.join(LOG_DIR, 'sector_targets.csv'), index=False)
print(f'Target list saved — {len(target_df)} stars to download')

## 5. Download Science Light Curves in Batches
Downloads all science targets. **This is the slow step — run overnight for full dataset.**

In [ ]:
def download_one(tic_id, save_dir, sector=None):
    """Download one light curve. Returns (success, n_points)."""
    save_path = os.path.join(save_dir, f'TIC_{tic_id}.fits')
    if os.path.exists(save_path):
        return True, 0   # Already downloaded

    # Try with sector first, then any sector
    search_kwargs_list = [
        dict(mission='TESS', sector=sector, exptime=120),
        dict(mission='TESS', sector=sector),
        dict(mission='TESS'),
    ]

    for kwargs in search_kwargs_list:
        try:
            sr = lk.search_lightcurve(f'TIC {tic_id}', **kwargs)
            if len(sr) > 0:
                lc = sr[0].download()
                lc.to_fits(save_path, overwrite=True)
                return True, len(lc)
        except:
            continue
    return False, 0


# ── MAIN DOWNLOAD LOOP ────────────────────────────────────────────────
tic_list     = target_df['tic_id'].tolist()
success_log  = []
fail_log     = []

# Check how many already downloaded
already_done = glob.glob(os.path.join(RAW_DIR, 'TIC_*.fits'))
done_ids     = set([os.path.basename(f).replace('TIC_','').replace('.fits','')
                    for f in already_done])
remaining    = [t for t in tic_list if str(t) not in done_ids]

print(f'Total targets    : {len(tic_list)}')
print(f'Already done     : {len(done_ids)}')
print(f'Remaining        : {len(remaining)}')
print(f'Estimated time   : ~{len(remaining)*3/60:.0f} minutes')
print()

for i, tic_id in enumerate(tqdm(remaining, desc='Downloading science data')):
    ok, npts = download_one(str(tic_id), RAW_DIR, sector=SECTOR)

    if ok:
        success_log.append({'tic_id': tic_id, 'n_points': npts})
    else:
        fail_log.append({'tic_id': tic_id})

    # Save progress every 100 stars
    if (i+1) % 100 == 0:
        pd.DataFrame(success_log).to_csv(
            os.path.join(LOG_DIR, 'download_progress.csv'), index=False
        )
        n_done = len(glob.glob(os.path.join(RAW_DIR, 'TIC_*.fits')))
        print(f'\n  Progress: {n_done} files saved, {len(fail_log)} failed')

    time.sleep(SLEEP_BETWEEN)

# Final summary
total_files = glob.glob(os.path.join(RAW_DIR, 'TIC_*.fits'))
print(f'\n✅ Download complete!')
print(f'   Total FITS files in raw/ : {len(total_files)}')
print(f'   Failed this session      : {len(fail_log)}')

pd.DataFrame(success_log).to_csv(
    os.path.join(LOG_DIR, 'download_success.csv'), index=False
)
pd.DataFrame(fail_log).to_csv(
    os.path.join(LOG_DIR, 'download_failed.csv'), index=False
)

## 6. Download Curated Labeled Dataset
Downloads labeled data from ExoFOP — the official TESS planet/EB/FP catalog.
This is equivalent to the 'curated dataset' mentioned in the problem statement.

In [ ]:
print('Downloading ExoFOP TOI catalog (official TESS labeled dataset)...')

# Official ExoFOP TOI catalog — thousands of labeled objects
toi_url = 'https://exofop.ipac.caltech.edu/tess/download_toi.php?sort=toi&output=csv'

try:
    toi_df = pd.read_csv(toi_url, comment='#')
    print(f'TOI catalog downloaded: {len(toi_df)} entries')
    print('Dispositions available:')
    print(toi_df['TFOPWG Disposition'].value_counts())
except Exception as e:
    print(f'Download failed: {e}')
    print('Trying backup URL...')
    # Backup — use requests
    r = requests.get(toi_url, timeout=60)
    from io import StringIO
    toi_df = pd.read_csv(StringIO(r.text), comment='#')
    print(f'TOI catalog loaded: {len(toi_df)} entries')

toi_df.to_csv(os.path.join(LOG_DIR, 'toi_catalog.csv'), index=False)
print('TOI catalog saved!')

## 7. Build Balanced Labeled Dataset

In [ ]:
# Map official dispositions to our 4 classes
label_map = {
    'KP' : 'planet',             # Known Planet
    'CP' : 'planet',             # Confirmed Planet
    'PC' : 'planet_candidate',   # Planet Candidate
    'FP' : 'false_positive',     # False Positive
    'EB' : 'eclipsing_binary',   # Eclipsing Binary
}

# Filter to known dispositions only
toi_labeled = toi_df[toi_df['TFOPWG Disposition'].isin(label_map)].copy()
toi_labeled['label']  = toi_labeled['TFOPWG Disposition'].map(label_map)
toi_labeled['tic_id'] = toi_labeled['TIC ID'].astype(str)

print('Available labeled samples per class:')
print(toi_labeled['label'].value_counts())
print()

# Build balanced dataset — N_PER_CLASS per label
balanced_parts = []
for label, group in toi_labeled.groupby('label'):
    n = min(N_PER_CLASS, len(group))
    balanced_parts.append(group.sample(n=n, random_state=42))

balanced_df = pd.concat(balanced_parts).reset_index(drop=True)
balanced_df = balanced_df[['tic_id', 'label']].copy()

print(f'Balanced training set:')
print(balanced_df['label'].value_counts())
print(f'Total: {len(balanced_df)} stars to download for training')

balanced_df.to_csv(os.path.join(LABELED_DIR, 'toi_labels_full.csv'), index=False)

## 8. Download Labeled Light Curves

In [ ]:
label_success = []
label_fail    = []

print(f'Downloading {len(balanced_df)} labeled light curves...')
print(f'Estimated time: ~{len(balanced_df)*3/60:.0f} minutes')
print()

for i, (_, row) in enumerate(tqdm(balanced_df.iterrows(),
                                   total=len(balanced_df),
                                   desc='Labeled data')):
    tic_id = str(row['tic_id'])
    label  = row['label']

    # Save with label in filename
    save_path = os.path.join(LABELED_DIR, f'TIC_{tic_id}_{label}.fits')

    if os.path.exists(save_path):
        label_success.append({'tic_id': tic_id, 'label': label, 'status': 'cached'})
        continue

    try:
        sr = lk.search_lightcurve(f'TIC {tic_id}', mission='TESS')
        if len(sr) == 0:
            label_fail.append({'tic_id': tic_id, 'label': label, 'reason': 'not found'})
            continue
        lc = sr[0].download()
        lc.to_fits(save_path, overwrite=True)
        label_success.append({'tic_id': tic_id, 'label': label, 'status': 'ok'})
    except Exception as e:
        label_fail.append({'tic_id': tic_id, 'label': label, 'reason': str(e)[:50]})

    # Save progress every 50
    if (i+1) % 50 == 0:
        pd.DataFrame(label_success).to_csv(
            os.path.join(LABELED_DIR, 'labels.csv'), index=False
        )
        print(f'  Progress: {len(label_success)} ok, {len(label_fail)} failed')

    time.sleep(SLEEP_BETWEEN)

# Final save
label_df = pd.DataFrame(label_success)
label_df.to_csv(os.path.join(LABELED_DIR, 'labels.csv'), index=False)

print(f'\n✅ Labeled download complete!')
print(f'   Downloaded : {len(label_success)}')
print(f'   Failed     : {len(label_fail)}')
print()
print('Label distribution downloaded:')
print(label_df['label'].value_counts())

## 9. Final Dataset Summary

In [ ]:
raw_files     = glob.glob(os.path.join(RAW_DIR,     'TIC_*.fits'))
labeled_files = glob.glob(os.path.join(LABELED_DIR, 'TIC_*.fits'))

print('=' * 55)
print('  FULL DATASET DOWNLOAD SUMMARY')
print('=' * 55)
print(f'  Science light curves     : {len(raw_files)}')
print(f'  Labeled light curves     : {len(labeled_files)}')
print(f'  Total FITS files         : {len(raw_files)+len(labeled_files)}')
print(f'  Approx disk usage        : ~{(len(raw_files)+len(labeled_files))*15}MB')
print('=' * 55)
print()
print('Next steps — run in order:')
print('  02_preprocessing.ipynb')
print('  03_bls_search.ipynb')
print('  04_feature_extraction.ipynb')
print('  05_model_training.ipynb')
print('  06_classification.ipynb')
print('  08_visualization.ipynb')

## 10. Plot Sample from Full Dataset

In [ ]:
# Plot 6 random science targets + 2 from each labeled class
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

# 6 random science targets
import random
sample_raw = random.sample(raw_files, min(6, len(raw_files)))

# 2 from each labeled class
sample_labeled = []
for label in ['planet','eclipsing_binary','false_positive','planet_candidate']:
    lfiles = glob.glob(os.path.join(LABELED_DIR, f'*_{label}.fits'))
    sample_labeled.extend(lfiles[:1])

all_samples = sample_raw + sample_labeled

for idx, fpath in enumerate(all_samples[:12]):
    ax = axes[idx]
    try:
        lc     = lk.read(fpath)
        flux   = lc.flux.value
        time   = lc.time.value
        mask   = np.isfinite(flux) & np.isfinite(time)
        fname  = os.path.basename(fpath)
        tic_id = fname.split('_')[1]

        # Detect label from filename
        if 'planet_candidate' in fname:   color, lbl = 'lime',   'PC'
        elif 'planet'          in fname:  color, lbl = 'green',  'Planet'
        elif 'eclipsing'       in fname:  color, lbl = 'red',    'EB'
        elif 'false'           in fname:  color, lbl = 'orange', 'FP'
        else:                             color, lbl = 'gray',   'Science'

        ax.plot(time[mask], flux[mask], '.', color=color,
                markersize=1, alpha=0.5)
        ax.set_title(f'TIC {tic_id}\n[{lbl}]  {mask.sum()} pts',
                     fontsize=8, color=color, fontweight='bold')
        ax.set_xlabel('Time (BTJD)', fontsize=7)
        ax.set_ylabel('Flux', fontsize=7)
    except Exception as e:
        ax.text(0.5, 0.5, str(e)[:40], ha='center', va='center',
                transform=ax.transAxes, fontsize=7)

for idx in range(len(all_samples), 12):
    axes[idx].set_visible(False)

plt.suptitle(f'Sample Light Curves — Full Dataset\n'
             f'Green=Planet  Red=EB  Orange=FP  Gray=Science',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plots/full_dataset_sample.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample plot saved!')

---
## ✅ Done!

### What Was Downloaded
| Data | Source | Purpose |
|---|---|---|
| Science LCs | TESS Sector 1, 2-min cadence | Run pipeline on |
| Labeled planets | ExoFOP KP+CP confirmed | Train classifier |
| Labeled EBs | ExoFOP EB disposition | Train classifier |
| Labeled FPs | ExoFOP FP disposition | Train classifier |
| Labeled PCs | ExoFOP PC disposition | Train classifier |

### Now Re-run the Full Pipeline
```
02 → 03 → 04 → 05 → 06 → 08
```

With 300+ labeled samples per class, expect:
- **RF accuracy: 88-95%**
- **4 classes: planet / EB / FP / planet_candidate**
- **Proper train/test split (80/20)**
- **Real confusion matrix and F1 scores**